# Project 8 — Semantic AI Search Engine

## Overview
We build a semantic search engine that understands **meaning** rather than just keywords. We compare:
- **TF-IDF search**: keyword frequency-based (traditional)
- **Semantic search**: embedding-based (modern)

```
Query → Embed → Vector Index → Top-k Results → Re-rank (optional)
```

## 1. Theory — Semantic Search

### TF-IDF
Term Frequency-Inverse Document Frequency represents documents as sparse vectors:
$$\text{TF-IDF}(t, d) = \underbrace{\frac{f_{t,d}}{\sum_k f_{k,d}}}_{\text{TF}} \times \underbrace{\log\frac{N}{df_t}}_{\text{IDF}}$$

where $f_{t,d}$ = count of term $t$ in doc $d$, $N$ = total docs, $df_t$ = docs containing term $t$.

**Limitation**: requires exact keyword match — "automobile" ≠ "car".

### Dense Retrieval (Semantic Search)
A bi-encoder maps query and documents to a shared dense vector space:
$$\text{score}(q, d) = \text{cosim}(f_Q(q), f_D(d)) = \frac{f_Q(q)^T f_D(d)}{\|f_Q(q)\| \|f_D(d)\|}$$

Semantic search understands synonyms, paraphrases, and conceptual relationships.

### Approximate Nearest Neighbour (ANN)
Brute-force search is $O(N \cdot D)$ which is slow for large $N$. ANN algorithms approximate results faster:

| Algorithm | Idea | Tradeoff |
|---|---|---|
| **HNSW** | Hierarchical graph structure | Very fast, high recall |
| **IVF** | Cluster into cells, search subset | Configurable accuracy vs speed |
| **LSH** | Random projections for bucketing | Simple, fast |

### Evaluation Metrics

**Recall@k**: fraction of relevant items found in top-k
$$\text{Recall@k} = \frac{|\text{Relevant} \cap \text{Retrieved}_k|}{|\text{Relevant}|}$$

**Mean Reciprocal Rank (MRR)**:
$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

where $\text{rank}_i$ is the rank of the first relevant result for query $i$.

**NDCG@k**: Normalised Discounted Cumulative Gain — accounts for position of relevant results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from typing import List, Dict, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style='whitegrid')
print('Imports done.')

## 2. Build Document Collection
We programmatically generate 100+ documents about ML/AI topics.

In [ ]:
# Topic categories and facts to generate documents from
TOPIC_DATA = {
    'supervised_learning': [
        'Supervised learning trains models on labelled data to predict outputs for new inputs.',
        'Classification assigns inputs to discrete categories like spam or not-spam.',
        'Regression predicts continuous values such as house prices or stock returns.',
        'Logistic regression uses the sigmoid function to model binary outcomes.',
        'Decision trees partition data by feature thresholds to reach leaf predictions.',
        'Support vector machines find the maximum margin hyperplane between classes.',
        'K-nearest neighbours classifies based on the majority class among k closest training points.',
        'Naive Bayes applies Bayes theorem assuming feature independence.',
        'Linear regression models the relationship between inputs and outputs as a straight line.',
        'Random forests combine many decision trees via bootstrap aggregation to reduce variance.',
    ],
    'unsupervised_learning': [
        'Unsupervised learning finds hidden patterns in data without labels.',
        'K-means clustering assigns each point to the nearest centroid iteratively.',
        'Principal component analysis reduces dimensionality by projecting to eigenvectors.',
        'DBSCAN clusters based on density, naturally finding arbitrarily-shaped clusters.',
        'Autoencoders compress data to a bottleneck and reconstruct it to learn representations.',
        'Gaussian mixture models represent data as a combination of multiple Gaussian distributions.',
        'Hierarchical clustering builds a dendrogram by merging closest clusters bottom-up.',
        't-SNE visualises high-dimensional data in two or three dimensions.',
        'UMAP is a faster dimensionality reduction technique that preserves global structure better than t-SNE.',
        'Self-supervised learning generates supervision signals from the data itself.',
    ],
    'neural_networks': [
        'Neural networks are computational graphs inspired by biological neurons.',
        'Backpropagation computes gradients by applying the chain rule backwards through the network.',
        'ReLU activation functions introduce non-linearity and avoid the vanishing gradient problem.',
        'Batch normalisation normalises layer activations to accelerate training.',
        'Dropout randomly zeros neurons during training to prevent overfitting.',
        'Residual connections add the input directly to the output, enabling very deep networks.',
        'Weight initialisation using He or Xavier methods prevents gradient explosion or vanishing.',
        'The universal approximation theorem states a single hidden layer can approximate any function.',
        'Convolutional layers extract local features using shared weight filters.',
        'Recurrent networks maintain hidden state to process sequential data.',
    ],
    'transformers': [
        'The transformer architecture relies on self-attention instead of convolution or recurrence.',
        'Multi-head attention allows the model to jointly attend to different representation subspaces.',
        'BERT uses bidirectional transformers pre-trained on masked language modelling.',
        'GPT uses unidirectional transformers trained with next-token prediction.',
        'Positional encoding injects position information into token representations.',
        'Layer normalisation stabilises training by normalising activations within a layer.',
        'Feed-forward sub-layers in transformers expand and compress feature dimensions.',
        'Attention scores represent how much one token attends to other tokens in context.',
        'Cross-attention in encoder-decoder models allows the decoder to attend to encoder outputs.',
        'Pre-training on large corpora followed by fine-tuning became the dominant NLP paradigm.',
    ],
    'optimization': [
        'Stochastic gradient descent updates parameters using one random sample at a time.',
        'Mini-batch gradient descent averages gradients over a small batch of samples.',
        'Adam optimizer combines momentum and adaptive learning rates for each parameter.',
        'Learning rate schedules reduce the step size over training to improve convergence.',
        'Gradient clipping prevents exploding gradients by capping gradient norms.',
        'Weight decay adds an L2 penalty to the loss to regularise model weights.',
        'Momentum accumulates gradients to dampen oscillations and accelerate convergence.',
        'AdaGrad adapts the learning rate for each parameter based on historical gradients.',
        'RMSProp divides the learning rate by an exponentially decaying average of squared gradients.',
        'Cosine annealing reduces the learning rate following a cosine curve to a minimum value.',
    ],
    'evaluation': [
        'Accuracy measures the fraction of correct predictions out of all predictions.',
        'Precision is the fraction of true positives among all positive predictions.',
        'Recall is the fraction of actual positives that the model correctly identifies.',
        'F1-score is the harmonic mean of precision and recall.',
        'AUC-ROC measures the area under the receiver operating characteristic curve.',
        'Mean squared error is the average squared difference between predictions and targets.',
        'Cross-validation averages performance over multiple train-validation splits.',
        'Confusion matrix shows counts of true and false positives and negatives.',
        'Perplexity measures how well a language model predicts a held-out test set.',
        'BLEU score evaluates text generation quality against reference translations.',
    ],
    'infrastructure': [
        'CUDA enables GPU-accelerated computation for matrix operations in deep learning.',
        'Distributed training splits model training across multiple GPUs or machines.',
        'Model checkpointing saves intermediate weights to resume training after failure.',
        'Quantisation reduces model precision from float32 to int8 to save memory and increase speed.',
        'Knowledge distillation trains a smaller student model to mimic a larger teacher model.',
        'ONNX provides an open format for representing trained machine learning models.',
        'Tensorboard visualises training metrics, model graphs, and embeddings.',
        'Docker containers package ML models with all dependencies for reproducible deployment.',
        'Model serving APIs expose trained models as HTTP endpoints for inference.',
        'Feature stores centralise feature computation and serving for ML pipelines.',
    ],
}

# Build document corpus
documents = []
doc_topics = []
for topic, facts in TOPIC_DATA.items():
    for fact in facts:
        documents.append(fact)
        doc_topics.append(topic)

print(f'Total documents: {len(documents)}')
print(f'Topics: {list(TOPIC_DATA.keys())}')

## 3. TF-IDF Search Engine

In [ ]:
class TFIDFSearch:
    """Keyword-based search using TF-IDF vectors."""

    def __init__(self):
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
        self.doc_vectors = None
        self.documents   = None

    def index(self, documents: List[str]):
        """Build TF-IDF index from documents."""
        self.documents   = documents
        self.doc_vectors = self.vectorizer.fit_transform(documents)  # sparse matrix
        print(f'TF-IDF index built: {self.doc_vectors.shape}')

    def search(self, query: str, k: int = 5) -> List[dict]:
        """Search for top-k documents by TF-IDF cosine similarity."""
        q_vec = self.vectorizer.transform([query])  # sparse query vector
        sims  = cosine_similarity(q_vec, self.doc_vectors)[0]  # (N,)

        top_k = np.argsort(sims)[::-1][:k]
        return [{
            'doc_id': int(i),
            'text':   self.documents[i],
            'score':  float(sims[i])
        } for i in top_k]


tfidf_engine = TFIDFSearch()
tfidf_engine.index(documents)

## 4. Semantic Search Engine

In [ ]:
from sentence_transformers import SentenceTransformer

class SemanticSearch:
    """Dense retrieval using sentence embeddings."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        print(f'Loading model: {model_name}')
        self.model     = SentenceTransformer(model_name)
        self.doc_embs  = None
        self.documents = None

    def index(self, documents: List[str]):
        """Embed all documents and store normalised vectors."""
        self.documents = documents
        print(f'Embedding {len(documents)} documents...')
        embs = self.model.encode(documents, show_progress_bar=True, convert_to_numpy=True)
        # L2-normalise for fast cosine via dot product
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        self.doc_embs = embs / (norms + 1e-10)
        print(f'Index built: {self.doc_embs.shape}')

    def search(self, query: str, k: int = 5) -> List[dict]:
        """Find top-k semantically similar documents."""
        q_emb = self.model.encode([query], convert_to_numpy=True)[0]
        q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-10)

        sims  = self.doc_embs @ q_emb  # dot product of normalised = cosine
        top_k = np.argsort(sims)[::-1][:k]

        return [{
            'doc_id': int(i),
            'text':   self.documents[i],
            'score':  float(sims[i])
        } for i in top_k]


semantic_engine = SemanticSearch()
semantic_engine.index(documents)

## 5. Side-by-Side Comparison

In [ ]:
TEST_QUERIES = [
    ('neural net training', 'neural_networks'),
    ('how to prevent overfitting', 'neural_networks'),
    ('cars and automobiles',  'supervised_learning'),  # semantic test — no exact keyword match
    ('speed up model inference', 'infrastructure'),
    ('measuring model quality', 'evaluation'),
]

print('=== SEARCH COMPARISON ===\n')
for query, expected_topic in TEST_QUERIES:
    tfidf_results  = tfidf_engine.search(query,   k=3)
    semantic_results = semantic_engine.search(query, k=3)

    print(f'Query: "{query}" (expected topic: {expected_topic})')
    print(f'  TF-IDF     : {tfidf_results[0]["text"][:70]}... ({tfidf_results[0]["score"]:.3f})')
    print(f'  Semantic   : {semantic_results[0]["text"][:70]}... ({semantic_results[0]["score"]:.3f})')
    print()

## 6. Performance Benchmark

In [ ]:
BENCHMARK_QUERIES = [q for q, _ in TEST_QUERIES]
N_RUNS = 20

# TF-IDF timing
t0 = time.time()
for _ in range(N_RUNS):
    for q in BENCHMARK_QUERIES:
        tfidf_engine.search(q, k=5)
tfidf_time = (time.time() - t0) / (N_RUNS * len(BENCHMARK_QUERIES)) * 1000

# Semantic timing (encoding is the bottleneck)
t0 = time.time()
for _ in range(N_RUNS):
    for q in BENCHMARK_QUERIES:
        semantic_engine.search(q, k=5)
semantic_time = (time.time() - t0) / (N_RUNS * len(BENCHMARK_QUERIES)) * 1000

print(f'TF-IDF average query latency:   {tfidf_time:.2f} ms')
print(f'Semantic average query latency: {semantic_time:.2f} ms')
print(f'\nNote: Semantic search is slower due to neural network query encoding.')
print(f'In production, use FAISS or another ANN library to reduce retrieval time to ~1ms at scale.')

## 7. Visualisation — Embedding Space

In [ ]:
from sklearn.manifold import TSNE

# Reduce embeddings to 2D with t-SNE for visualisation
tsne = TSNE(n_components=2, random_state=42, perplexity=20, n_iter=500)
emb_2d = tsne.fit_transform(semantic_engine.doc_embs)

# Map topics to colours
topic_list   = list(TOPIC_DATA.keys())
colour_map   = plt.cm.get_cmap('tab10', len(topic_list))
topic_colors = [colour_map(topic_list.index(t)) for t in doc_topics]

plt.figure(figsize=(10, 8))
for i, topic in enumerate(topic_list):
    mask = [t == topic for t in doc_topics]
    pts  = emb_2d[mask]
    plt.scatter(pts[:, 0], pts[:, 1], label=topic, alpha=0.7, s=60)

# Plot query embeddings
for query, _ in TEST_QUERIES:
    q_emb = semantic_engine.model.encode([query], convert_to_numpy=True)[0]
    q_norm = q_emb / np.linalg.norm(q_emb)
    # Project query into same 2D space approximately
    # (Note: this is approximate since t-SNE is not invertible)

plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1))
plt.title('t-SNE Visualisation of Document Embeddings by Topic')
plt.tight_layout()
plt.show()

## 8. Evaluation — Recall@5

In [ ]:
# Ground truth: for each query, documents from the expected topic are relevant
def recall_at_k(results: List[dict], relevant_ids: List[int], k: int) -> float:
    """Recall@k: fraction of relevant docs found in top-k."""
    retrieved = {r['doc_id'] for r in results[:k]}
    return len(retrieved & set(relevant_ids)) / max(len(relevant_ids), 1)


# Build ground truth: indices of all docs belonging to expected topic
tfidf_recalls, semantic_recalls = [], []

for query, expected_topic in TEST_QUERIES:
    relevant_ids = [i for i, t in enumerate(doc_topics) if t == expected_topic]

    tfidf_r  = tfidf_engine.search(query,   k=5)
    sem_r    = semantic_engine.search(query, k=5)

    tr = recall_at_k(tfidf_r,  relevant_ids, k=5)
    sr = recall_at_k(sem_r,    relevant_ids, k=5)

    tfidf_recalls.append(tr)
    semantic_recalls.append(sr)

    print(f'Query: "{query[:40]}"  TF-IDF R@5={tr:.2f}  Semantic R@5={sr:.2f}')

print(f'\nMean Recall@5  TF-IDF: {np.mean(tfidf_recalls):.3f}  Semantic: {np.mean(semantic_recalls):.3f}')

# Bar chart comparison
x      = np.arange(len(TEST_QUERIES))
labels = [q[:20] + '...' for q, _ in TEST_QUERIES]

plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, tfidf_recalls,    0.4, label='TF-IDF', color='steelblue')
plt.bar(x + 0.2, semantic_recalls, 0.4, label='Semantic', color='darkorange')
plt.xticks(x, labels, rotation=15, ha='right')
plt.ylabel('Recall@5')
plt.title('TF-IDF vs Semantic Search: Recall@5')
plt.legend()
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

## Summary

| Method | Strengths | Weaknesses |
|---|---|---|
| TF-IDF | Fast, no GPU, interpretable | Exact keyword match only, misses synonyms |
| Semantic | Understands meaning, handles synonyms | Slower encoding, needs embedding model |

**Production recommendation**: Use semantic search for recall-critical applications. Use TF-IDF as a fast pre-filter in a hybrid system. For large corpora (>1M docs), replace the numpy brute-force with FAISS for sub-millisecond latency.